# Stage 01 · Paper Full CUDA

本 notebook 只负责 Colab 环境准备，并调用 scripts/01_Paper_Full_Cuda.py。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "01_Paper_Full_Cuda"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "main"
REPO_ROOT = Path("/content/ceg_wm_workspace")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_project_root"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
SCRIPT_PATH = REPO_ROOT / "scripts" / "01_Paper_Full_Cuda.py"
ATTESTATION_ENV_PATH = DRIVE_PROJECT_ROOT / "secrets" / "attestation_env.json"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
SEMANTIC_MODEL_SOURCE_URLS = {
    "inspyrenet": "https://github.com/plemeri/InSPyReNet/releases/download/v1.0.0/InSPyReNet.pth",
}


def run_checked(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)



def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path



def load_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)



def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))


In [ ]:
from google.colab import drive


drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
ensure_dir(DRIVE_PROJECT_ROOT)
ensure_dir(HF_HOME)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_url = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    ).stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print_json(
    "repo_binding",
    {
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "config_path": str(CONFIG_PATH),
        "script_path": str(SCRIPT_PATH),
    },
)


In [ ]:
import urllib.request

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

from huggingface_hub import snapshot_download
from scripts.notebook_runtime_common import (
    build_directory_digest_summary,
    collect_weight_summary,
    load_yaml_mapping,
    resolve_model_identity,
)

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
MODEL_IDENTITY = resolve_model_identity(CFG_OBJ)
MASK_CFG = CFG_OBJ.get("mask", {}) if isinstance(CFG_OBJ.get("mask"), dict) else {}
PROMPT_SOURCE_PATH = REPO_ROOT / str(CFG_OBJ.get("inference_prompt_file", ""))
WEIGHT_PATH = REPO_ROOT / str(MASK_CFG.get("semantic_model_path", ""))
WEIGHT_SOURCE = str(MASK_CFG.get("semantic_model_source", ""))
WEIGHT_URL = SEMANTIC_MODEL_SOURCE_URLS.get(WEIGHT_SOURCE)
ensure_dir(WEIGHT_PATH.parent)

if ATTESTATION_ENV_PATH.exists():
    env_payload = load_json(ATTESTATION_ENV_PATH)
    for key, value in env_payload.items():
        os.environ[str(key)] = str(value)

if not WEIGHT_PATH.exists():
    if not WEIGHT_URL:
        raise RuntimeError(f"unsupported semantic_model_source for notebook bootstrap: {WEIGHT_SOURCE}")
    urllib.request.urlretrieve(WEIGHT_URL, WEIGHT_PATH)

MODEL_SNAPSHOT_PATH = snapshot_download(
    repo_id=str(MODEL_IDENTITY["model_id"]),
    revision=None if MODEL_IDENTITY["revision"] == "<absent>" else str(MODEL_IDENTITY["revision"]),
    cache_dir=str(HF_HOME),
)
MODEL_DOWNLOAD_SUMMARY = build_directory_digest_summary(Path(MODEL_SNAPSHOT_PATH))
WEIGHT_DOWNLOAD_SUMMARY = collect_weight_summary(REPO_ROOT, CFG_OBJ)
CONFIG_BINDINGS = {
    "prompt_source_path": str(PROMPT_SOURCE_PATH),
    "prompt_exists": PROMPT_SOURCE_PATH.exists(),
    "model_id": MODEL_IDENTITY["model_id"],
    "hf_revision": MODEL_IDENTITY["revision"],
    "model_snapshot_path": MODEL_SNAPSHOT_PATH,
    "semantic_model_path": str(WEIGHT_PATH),
    "semantic_model_source": WEIGHT_SOURCE,
}
print_json("config_bindings", CONFIG_BINDINGS)
print_json("model_download_summary", MODEL_DOWNLOAD_SUMMARY)
print_json("weight_download_summary", WEIGHT_DOWNLOAD_SUMMARY)
run_checked(["nvidia-smi"])


In [ ]:
from pathlib import Path

from scripts.notebook_runtime_common import make_stage_run_id

STAGE_RUN_ID = make_stage_run_id(NOTEBOOK_NAME)
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--config",
    str(CONFIG_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--stage-run-id",
    STAGE_RUN_ID,
]
run_checked(COMMAND, cwd=REPO_ROOT, env=os.environ.copy())

SUMMARY_PATH = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / STAGE_RUN_ID / "stage_summary.json"
STAGE_SUMMARY = load_json(SUMMARY_PATH)
STAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["stage_manifest_path"]))
PACKAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["package_manifest_path"]))
print_json("stage_summary", STAGE_SUMMARY)
print_json("stage_manifest", STAGE_MANIFEST)
print_json("package_manifest", PACKAGE_MANIFEST)


In [ ]:
from scripts.notebook_runtime_common import collect_missing_file_entries, compute_file_sha256

PACKAGE_PATH = Path(STAGE_SUMMARY["package_path"])
REQUIRED_STAGE_FILES = {
    "stage_manifest": Path(STAGE_SUMMARY["stage_manifest_path"]),
    "package_manifest": Path(STAGE_SUMMARY["package_manifest_path"]),
    "package_zip": PACKAGE_PATH,
    "embed_record": Path(STAGE_MANIFEST["records"]["embed_record"]["path"]),
    "detect_record": Path(STAGE_MANIFEST["records"]["detect_record"]["path"]),
    "calibration_record": Path(STAGE_MANIFEST["records"]["calibration_record"]["path"]),
    "evaluate_record": Path(STAGE_MANIFEST["records"]["evaluate_record"]["path"]),
    "thresholds_artifact": Path(STAGE_MANIFEST["thresholds_path"]),
    "evaluation_report": Path(STAGE_MANIFEST["evaluation_report_path"]),
    "run_closure": Path(STAGE_MANIFEST["run_closure_path"]),
    "workflow_summary": Path(STAGE_MANIFEST["workflow_summary_path"]),
    "runtime_config_snapshot": Path(STAGE_MANIFEST["runtime_config_snapshot_path"]),
    "prompt_snapshot": Path(STAGE_MANIFEST["prompt_snapshot_path"]),
}
MISSING_STAGE_FILES = collect_missing_file_entries(REQUIRED_STAGE_FILES)
PACKAGE_SHA256 = compute_file_sha256(PACKAGE_PATH)
if PACKAGE_SHA256 != PACKAGE_MANIFEST["package_sha256"]:
    raise RuntimeError(
        f"package sha256 mismatch: manifest={PACKAGE_MANIFEST['package_sha256']} actual={PACKAGE_SHA256}"
    )
if not str(STAGE_MANIFEST["config_source_path"]).endswith("/configs/default.yaml"):
    raise RuntimeError(f"config_source_path is not bound to default.yaml: {STAGE_MANIFEST['config_source_path']}")
if MISSING_STAGE_FILES:
    raise FileNotFoundError(f"stage 01 validation failed, missing files: {MISSING_STAGE_FILES}")
VALIDATION_RESULT = {
    "stage_name": NOTEBOOK_NAME,
    "stage_run_id": STAGE_RUN_ID,
    "package_sha256": PACKAGE_SHA256,
    "missing_files": MISSING_STAGE_FILES,
    "validated_paths": {key: str(value) for key, value in REQUIRED_STAGE_FILES.items()},
    "status": "ok",
}
print_json("validation_result", VALIDATION_RESULT)


In [ ]:
from scripts.notebook_runtime_common import summarize_manifest_fields, tail_text_file

LOG_ROOT = Path(STAGE_SUMMARY["log_root"])
LOG_FILES = sorted(LOG_ROOT.rglob("*.log"), key=lambda item: item.stat().st_mtime if item.exists() else 0)
LATEST_LOG_PATH = LOG_FILES[-1] if LOG_FILES else None
DIAGNOSTIC_RESULT = {
    "stage_run_id": STAGE_RUN_ID,
    "run_root": STAGE_SUMMARY["run_root"],
    "log_root": STAGE_SUMMARY["log_root"],
    "runtime_state_root": STAGE_SUMMARY["runtime_state_root"],
    "export_root": STAGE_SUMMARY["export_root"],
    "missing_files": VALIDATION_RESULT["missing_files"],
    "latest_log_path": str(LATEST_LOG_PATH) if LATEST_LOG_PATH else "<absent>",
    "latest_log_tail": tail_text_file(LATEST_LOG_PATH, max_lines=20) if LATEST_LOG_PATH else [],
    "stage_manifest_summary": summarize_manifest_fields(
        STAGE_MANIFEST,
        [
            "stage_name",
            "stage_run_id",
            "config_source_path",
            "prompt_snapshot_path",
            "runtime_config_snapshot_path",
            "thresholds_path",
            "workflow_summary_path",
        ],
    ),
    "package_manifest_summary": summarize_manifest_fields(
        PACKAGE_MANIFEST,
        [
            "package_filename",
            "package_sha256",
            "package_created_at",
            "package_contents_index_path",
            "package_manifest_version",
        ],
    ),
    "status": "ok" if not VALIDATION_RESULT["missing_files"] else "diagnostic_required",
}
print_json("diagnostic_result", DIAGNOSTIC_RESULT)
